# Solutions — Advanced Iterators: itertools, Windows, Sort-Merge Joins, Lazy Pipelines

Exercise notebook: `10_advanced_python/01_generators_itertools.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Running maximum

**Working solution**

In [ ]:
from itertools import accumulate

def running_max(values):
    return accumulate(values, max)

**Explanation**

`accumulate(iterable, func)` is a lazy fold that emits every intermediate result — running sums (default), maxima, products, balances.

**Complexity:** O(n), O(1) memory.

**Common mistakes**

- Calling `max(values[:i+1])` for every i (O(n²)).

## Exercise 2 — 🟡 Medium · Sliding windows

**Working solution**

In [ ]:
from collections import deque

def windowed(iterable, n):
    if n < 1:
        raise ValueError("n must be >= 1")
    def gen():
        window = deque(maxlen=n)
        for item in iterable:
            window.append(item)
            if len(window) == n:
                yield tuple(window)
    return gen()

**Explanation**

A bounded deque drops the oldest element automatically. Wrapping the generator lets `ValueError` fire at call time.

**Complexity:** O(n·k) for k windows (tuple copies).

**Common mistakes**

- Slicing a list repeatedly — not lazy and O(n·k) copies anyway.

## Exercise 3 — 🟡 Medium · Run-length grouping with groupby

**Working solution**

In [ ]:
from itertools import groupby
from operator import itemgetter

def status_runs(events):
    runs = []
    for status, group in groupby(events, key=itemgetter("status")):
        items = list(group)
        runs.append((status, len(items), items[0]["ts"], items[-1]["ts"]))
    return runs

**Explanation**

`itertools.groupby` groups **consecutive** equal keys — perfect for run-length encoding (uptime runs, status streaks), and the reason it needs sorted input for SQL-style grouping.

**Complexity:** O(n).

**Common mistakes**

- Expecting `groupby` to merge non-adjacent groups.

## Exercise 4 — 🟠 Hard · Streaming sort-merge join

**Working solution**

In [ ]:
from itertools import groupby

def _groups(rows, key):
    previous = None
    for k, grp in groupby(rows, key=key):
        if previous is not None and k < previous:
            raise ValueError(f"input not sorted: {k!r} after {previous!r}")
        previous = k
        yield k, list(grp)

def merge_join(left, right, key):
    lg, rg = _groups(left, key), _groups(right, key)
    lk, lrows = next(lg, (None, None))
    rk, rrows = next(rg, (None, None))
    while lrows is not None and rrows is not None:
        if lk < rk:
            lk, lrows = next(lg, (None, None))
        elif rk < lk:
            rk, rrows = next(rg, (None, None))
        else:
            for l in lrows:
                for r in rrows:
                    yield l, r
            lk, lrows = next(lg, (None, None))
            rk, rrows = next(rg, (None, None))

**Explanation**

Two cursors advance over sorted streams; equal keys emit pairs. This is the **sort-merge join** Spark uses for large-large joins (after shuffling both sides by key):
memory per key group, not per table. `next(gen, default)` avoids `StopIteration` handling.

**Complexity:** O(n + m + output).

**Common mistakes**

- Loading one side into a dict — that's a hash join (fine when it fits in memory, which is the point of choosing).
- Advancing only one side on a match → infinite loop.

## Challenge — 🔴 Very Hard · A lazy, chainable pipeline API (mini RDD)

**Working solution**

In [ ]:
from itertools import chain, islice

class Stream:
    def __init__(self, source, _ops=()):
        self._source = source
        self._ops = tuple(_ops)

    def _with(self, op):
        return Stream(self._source, self._ops + (op,))

    # ----- transformations (lazy) -----
    def map(self, f):
        return self._with(lambda it: map(f, it))

    def filter(self, p):
        return self._with(lambda it: filter(p, it))

    def flat_map(self, f):
        return self._with(lambda it: chain.from_iterable(map(f, it)))

    def distinct(self, key=None):
        def op(it):
            seen = set()
            for x in it:
                k = key(x) if key else x
                if k not in seen:
                    seen.add(k)
                    yield x
        return self._with(op)

    def batch(self, n):
        def op(it):
            it = iter(it)
            while chunk := list(islice(it, n)):
                yield chunk
        return self._with(op)

    def take(self, n):
        return self._with(lambda it: islice(it, n))

    # ----- execution -----
    def __iter__(self):
        it = self._source() if callable(self._source) else self._source
        for op in self._ops:
            it = op(it)
        return iter(it)

    def collect(self):
        return list(self)

    def count(self):
        return sum(1 for _ in self)

    def reduce(self, f, initial):
        acc = initial
        for x in self:
            acc = f(acc, x)
        return acc

    def first(self, default=None):
        return next(iter(self), default)

    def group_by_key(self, key, value=lambda x: x):
        groups = {}
        for x in self:
            groups.setdefault(key(x), []).append(value(x))
        return groups

**Explanation**

Each transformation records a function `iterator -> iterator`; nothing executes until an action builds the iterator chain. That's precisely Spark's model: transformations build a
lineage (DAG), actions trigger execution, and re-running an action recomputes from the source unless you cache. Because each stage is a generator, `take(3)` stops pulling after 3 items.

**Complexity:** O(items actually pulled) per action.

**Common mistakes**

- Doing the work inside `map()` immediately (eager).
- Mutating `self._ops` instead of returning a new Stream — branches would interfere.